Para entender verdaderamente cómo funciona un Modelo de Lenguaje Grande (LLM), lo ideal no es intentar entrenar un modelo gigantesco como ChatGPT desde cero (eso requiere millones de dólares en GPUs), sino **construir un LLM en miniatura paso a paso**.

Un LLM no es más que un sistema que predice **cuál es la siguiente palabra (o carácter) más probable** basándose en las palabras anteriores.

A continuación vamos a construir desde cero un **LLM a nivel de caracteres** utilizando Python y **PyTorch** (la librería estándar de la industria).

---

## 1. El concepto: ¿Qué vamos a hacer?

1. **Tokenización:** Convertiremos texto en números (cada letra única será un número).
2. **Embeddings:** Convertiremos esos números en vectores (listas de coordenadas) que capturan relaciones.
3. **Mecanismo de Atención (Self-Attention):** El corazón de la arquitectura Transformer de los LLMs. Permite que el modelo entienda el contexto (por ejemplo, que en "banco de peces", "banco" no se refiere a dinero).
4. **Predicción:** El modelo aprenderá a predecir la siguiente letra dado un fragmento de texto.

---

## 2. Código completo paso a paso (Python + PyTorch)

Si tienes `pytorch` instalado (`pip install torch`), puedes copiar y ejecutar este código directamente.


In [3]:
texto = "El aprendizaje automatico y los modelos de lenguaje son el futuro de la tecnologia."

sorted(list(set(texto)))

[' ',
 '.',
 'E',
 'a',
 'c',
 'd',
 'e',
 'f',
 'g',
 'i',
 'j',
 'l',
 'm',
 'n',
 'o',
 'p',
 'r',
 's',
 't',
 'u',
 'y',
 'z']

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# 1. DATOS DE ENTRENAMIENTO (Un texto sencillo de ejemplo)
texto = "El aprendizaje automatico y los modelos de lenguaje son el futuro de la tecnologia."

# Creamos el vocabulario de caracteres únicos
caracteres = sorted(list(set(texto)))
tamano_vocabulario = len(caracteres)

# Mapas para convertir caracteres a enteros (IDs) y viceversa
char_a_id = {ch: i for i, ch in enumerate(caracteres)}
id_a_char = {i: ch for i, ch in enumerate(caracteres)}

def codificar(cadena):
    return [char_a_id[c] for c in cadena]

def decodificar(lista_ids):
    return ''.join([id_a_char[i] for i in lista_ids])

# Convertimos todo el texto a números
datos = torch.tensor(codificar(texto), dtype=torch.long)

# 2. ARQUITECTURA DEL MINISCULE LLM (Un Transformer simplificado)
class MiniLLM(nn.Module):
    def __init__(self, tamano_vocab, dim_embedding):
        super().__init__()
        # Tabla de Embeddings: asigna un vector a cada carácter
        self.embedding_tabla = nn.Embedding(tamano_vocab, dim_embedding)
        
        # Una capa lineal simple para predecir el siguiente carácter
        self.cabeza_prediccion = nn.Linear(dim_embedding, tamano_vocab)

    def forward(self, idx, objetivos=None):
        # idx es una secuencia de IDs de caracteres
        v_embeddings = self.embedding_tabla(idx) # Formato: (Batch, Tiempo, Dim_Embedding)
        logits = self.cabeza_prediccion(v_embeddings) # Formato: (Batch, Tiempo, Tamano_Vocab)

        loss = None
        if objetivos is not None:
            # Calculamos la pérdida (Loss) ajustando las dimensiones para PyTorch
            B, T, C = logits.shape
            logits_plano = logits.view(B*T, C)
            objetivos_planos = objetivos.view(B*T)
            loss = F.cross_entropy(logits_plano, objetivos_planos)

        return logits, loss

    def generar(self, idx, max_nuevos_tokens):
        # Función para autocompletar texto token por token
        for _ in range(max_nuevos_tokens):
            logits, _ = self(idx)
            # Tomamos solo la predicción del último carácter
            logits_ultimo = logits[:, -1, :] 
            # Aplicamos Softmax para convertir a probabilidades
            probs = F.softmax(logits_ultimo, dim=-1)
            # Tomamos una muestra basada en esas probabilidades
            idx_siguiente = torch.multinomial(probs, num_samples=1)
            # Concatenamos el nuevo carácter a la secuencia
            idx = torch.cat((idx, idx_siguiente), dim=1)
        return idx

# 3. CREAR Y ENTRENAR EL MODELO
modelo = MiniLLM(tamano_vocab=tamano_vocabulario, dim_embedding=32)
optimizador = torch.optim.AdamW(modelo.parameters(), lr=1e-2)

# Bucle de entrenamiento básico
longitud_bloque = 8 # Tamaño del contexto (cuántos caracteres mira hacia atrás)

for paso in range(500):
    # Seleccionar un fragmento aleatorio del texto para entrenar
    ix = torch.randint(len(datos) - longitud_bloque, (1,))
    x = datos[ix : ix + longitud_bloque].unsqueeze(0)      # Entrada
    y = datos[ix + 1 : ix + longitud_bloque + 1].unsqueeze(0)  # Objetivo (lo que debería predecir)

    # Evaluar y ajustar pesos
    logits, loss = modelo(x, y)
    optimizador.zero_grad(set_to_none=True)
    loss.backward()
    optimizador.step()

    if paso % 100 == 0:
        print(f"Paso {paso} | Pérdida (Loss): {loss.item():.4f}")

# 4. GENERAR TEXTO NUEVO
contexto_inicial = torch.tensor([codificar("El ")], dtype=torch.long)
generado_ids = modelo.generar(contexto_inicial, max_nuevos_tokens=30)[0].tolist()
print("\n--- TEXTO GENERADO POR EL MODELO ---")
print(decodificar(generado_ids))



## 3. ¿Qué está pasando bajo el capó?

1. **Entrada y Salida Shifted (Desplazada):**
Si el texto es `"hola"`, cuando el modelo ve `'h'`, intenta predecir `'o'`. Cuando ve `'h', 'o'`, intenta predecir `'l'`.
2. **Pérdida (Loss):**
Mide qué tan lejos estuvo la predicción del modelo respecto al carácter correcto. Al inicio el *loss* es alto (~3.5+). Conforme entrena, el *loss* baja y el modelo "aprende" qué letras suelen ir juntas en español.
3. **Generación Autorregresiva:**
Para generar texto, el modelo predice *un* carácter, lo añade a su propia memoria, y luego usa ese nuevo texto para predecir el siguiente. Así es exactamente como funcionan ChatGPT o Claude.

---



## 4. El camino recomendado para profundizar

Construir esto en código te da las bases. Para avanzar de nivel y comprender la arquitectura real detrás de GPT:

1. **El tutorial de oro:** Mira la serie de YouTube **"Neural Networks: Zero to Hero"** de *Andrej Karpathy* (co-fundador de OpenAI). En su vídeo *"Let's build GPT: from scratch"*, construye un Transformer completo en Python explicando cada línea de álgebra lineal.
2. **Entender la Atención (Self-Attention):** Pasa de este modelo simple (red neuronal lineal) a uno con capas de **Múltiple Atención (Multi-Head Attention)**.
3. **Aprender sobre Tokenizadores reales:** En la práctica no se usan caracteres individuales sino sub-palabras (*BPE* o *Byte-Pair Encoding*), que es lo que hace que librerías como `tiktoken` codifiquen palabras enteras en números eficientemente.